# Cryptocurrency Volatility Prediction
This notebook implements data preprocessing, EDA, feature engineering, model training, and evaluation.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings("ignore")


## Load Dataset

In [ ]:

# Replace with your dataset path
df = pd.read_csv("crypto_data.csv")
print("Shape of dataset:", df.shape)
df.head()


## Data Cleaning

In [ ]:

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

df = df.dropna()
print("After removing missing values:", df.shape)


## Feature Engineering

In [ ]:

df['return'] = df['close'].pct_change()
df['volatility_7'] = df['return'].rolling(window=7).std()
df['volatility_14'] = df['return'].rolling(window=14).std()
df['ma_7'] = df['close'].rolling(window=7).mean()
df['ma_14'] = df['close'].rolling(window=14).mean()
df['liquidity_ratio'] = df['volume'] / df['market_cap']

df = df.dropna()
df.head()


## Exploratory Data Analysis

In [ ]:

plt.figure(figsize=(6,4))
sns.histplot(df['volatility_7'], bins=50, kde=True)
plt.title("Distribution of 7-Day Volatility")
plt.show()

plt.figure(figsize=(10,8))
corr = df[['open','high','low','close','volume','market_cap',
           'volatility_7','volatility_14','liquidity_ratio']].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()


## Prepare Features and Target

In [ ]:

target = 'volatility_7'

features = [
    'open', 'high', 'low', 'close',
    'volume', 'market_cap',
    'ma_7', 'ma_14',
    'volatility_14',
    'liquidity_ratio'
]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


## Feature Scaling

In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Model Training

In [ ]:

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train_scaled, y_train)
print("Model training completed.")


## Model Evaluation

In [ ]:

y_pred = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Evaluation Metrics:")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)


## Actual vs Predicted

In [ ]:

plt.figure(figsize=(10,5))
plt.plot(y_test.values[:200], label='Actual Volatility')
plt.plot(y_pred[:200], label='Predicted Volatility')
plt.legend()
plt.title("Actual vs Predicted Volatility (First 200 Test Points)")
plt.show()


## Feature Importance

In [ ]:

importances = pd.Series(model.feature_importances_, index=features)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(8,5))
importances.plot(kind='bar')
plt.title("Feature Importance")
plt.show()

importances
